In [1]:
# Importing libraries
import pandas as pd
import numpy as np
import seaborn as sns
import time
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler, LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.cluster import KMeans
from sklearn.metrics import (
    accuracy_score, confusion_matrix, ConfusionMatrixDisplay,
    RocCurveDisplay, auc, precision_score, recall_score,
    f1_score, roc_curve, classification_report, roc_auc_score
)
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.feature_selection import RFE
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from xgboost import XGBClassifier

In [49]:
#We already have a training set and a test set frm the data source. we merge the datasets to clean
#them only once
# Load the training set
train_df = pd.read_parquet('UNSW_NB15_training-set.parquet')
test_df = pd.read_parquet('UNSW_NB15_testing-set.parquet')

# Concatenate the training and test sets
df = pd.concat([train_df, test_df], ignore_index=True)

df.info()

df.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 257673 entries, 0 to 257672
Data columns (total 36 columns):
 #   Column             Non-Null Count   Dtype   
---  ------             --------------   -----   
 0   dur                257673 non-null  float32 
 1   proto              257673 non-null  object  
 2   service            257673 non-null  category
 3   state              257673 non-null  object  
 4   spkts              257673 non-null  int16   
 5   dpkts              257673 non-null  int16   
 6   sbytes             257673 non-null  int32   
 7   dbytes             257673 non-null  int32   
 8   rate               257673 non-null  float32 
 9   sload              257673 non-null  float32 
 10  dload              257673 non-null  float32 
 11  sloss              257673 non-null  int16   
 12  dloss              257673 non-null  int16   
 13  sinpkt             257673 non-null  float32 
 14  dinpkt             257673 non-null  float32 
 15  sjit               257673 non-null

,dur,proto,service,state,spkts,dpkts,sbytes,dbytes,rate,sload,...,trans_depth,response_body_len,ct_src_dport_ltm,ct_dst_sport_ltm,is_ftp_login,ct_ftp_cmd,ct_flw_http_mthd,is_sm_ips_ports,attack_cat,label
0,0.000011,udp,-,INT,2,0,496,0,90909.09375,180363632.0,...,0,0,1,1,0,0,0,0,Normal,0
1,0.000008,udp,-,INT,2,0,1762,0,125000.00000,881000000.0,...,0,0,1,1,0,0,0,0,Normal,0
2,0.000005,udp,-,INT,2,0,1068,0,200000.00000,854400000.0,...,0,0,1,1,0,0,0,0,Normal,0
3,0.000006,udp,-,INT,2,0,900,0,166666.65625,600000000.0,...,0,0,2,1,0,0,0,0,Normal,0
4,0.000010,udp,-,INT,2,0,2126,0,100000.00000,850400000.0,...,0,0,2,1,0,0,0,0,Normal,0


In [50]:
df = df[df['service'] != '-']

In [51]:
df1=df.copy() # To be used later

#drop irrelevant columns, hence id and the target column attack_category
drop_column = ["attack_cat"]
df.drop(drop_column, axis=1, inplace=True)

In [52]:
df_categorical = df.select_dtypes(exclude=[np.number])
label = LabelEncoder()
print(df_categorical.columns)
for feature in df_categorical.columns:
    df[feature] = label.fit_transform(df[feature])

Index(['proto', 'service', 'state'], dtype='object')


In [53]:
df

,dur,proto,service,state,spkts,dpkts,sbytes,dbytes,rate,sload,...,dmean,trans_depth,response_body_len,ct_src_dport_ltm,ct_dst_sport_ltm,is_ftp_login,ct_ftp_cmd,ct_flw_http_mthd,is_sm_ips_ports,label
35,0.983874,0,4,2,10,8,816,1172,17.278635,5.976375e+03,...,147,1,184,1,1,0,0,1,0,0
40,1.535254,0,4,2,10,10,826,1266,12.375802,3.876883e+03,...,127,1,187,1,1,0,0,1,0,0
45,1.059359,0,4,2,10,8,830,1134,16.047441,5.641147e+03,...,142,1,165,1,1,0,0,1,0,0
49,0.990548,0,4,2,10,10,804,1414,19.181301,5.847268e+03,...,141,1,261,1,1,0,0,1,0,0
72,1.303518,0,4,2,12,8,898,1120,14.575939,5.057084e+03,...,140,1,157,1,1,0,0,1,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
257667,0.000006,1,1,3,2,0,114,0,166666.656250,7.600000e+07,...,0,0,0,33,17,0,0,0,0,1
257668,0.000009,1,1,3,2,0,114,0,111111.109375,5.066666e+07,...,0,0,0,24,13,0,0,0,0,1
257670,0.000009,1,1,3,2,0,114,0,111111.109375,5.066666e+07,...,0,0,0,3,3,0,0,0,0,1
257671,0.000009,1,1,3,2,0,114,0,111111.109375,5.066666e+07,...,0,0,0,30,14,0,0,0,0,1


In [61]:

scaler = StandardScaler()
# Split Data into Features (X) and Target (y)
label_encoder = LabelEncoder()
df['label'] = label_encoder.fit_transform(df['label'])
X = df.drop(['label'], axis=1,errors='ignore')
y = df['label']

numerical_columns = X.select_dtypes(include=[np.number]).columns


# Df & df1
X[numerical_columns] = scaler.fit_transform(X[numerical_columns])

X[numerical_columns]

# import joblib
# joblib.dump(scaler,'label1.joblib')

,dur,proto,service,state,spkts,dpkts,sbytes,dbytes,rate,sload,...,smean,dmean,trans_depth,response_body_len,ct_src_dport_ltm,ct_dst_sport_ltm,is_ftp_login,ct_ftp_cmd,ct_flw_http_mthd,is_sm_ips_ports
35,0.045627,-1.203937,0.720069,-0.620894,-0.054516,-0.069046,-0.052843,-0.071704,-0.632651,-0.617410,...,-0.209306,0.116395,0.747477,-0.056250,-0.770080,-0.825661,-0.16555,-0.165479,0.724771,0.0
40,0.182081,-1.203937,0.720069,-0.620894,-0.054516,-0.054114,-0.052801,-0.071172,-0.632680,-0.617437,...,-0.204513,0.035370,0.747477,-0.056210,-0.770080,-0.825661,-0.16555,-0.165479,0.724771,0.0
45,0.064308,-1.203937,0.720069,-0.620894,-0.054516,-0.069046,-0.052784,-0.071918,-0.632658,-0.617414,...,-0.204513,0.096138,0.747477,-0.056509,-0.770080,-0.825661,-0.16555,-0.165479,0.724771,0.0
49,0.047279,-1.203937,0.720069,-0.620894,-0.054516,-0.054114,-0.052894,-0.070336,-0.632639,-0.617412,...,-0.218891,0.092087,0.747477,-0.055202,-0.770080,-0.825661,-0.16555,-0.165479,0.724771,0.0
72,0.124732,-1.203937,0.720069,-0.620894,-0.043517,-0.069046,-0.052498,-0.071997,-0.632667,-0.617422,...,-0.242855,0.088036,0.747477,-0.056618,-0.770080,-0.825661,-0.16555,-0.165479,0.724771,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
257667,-0.197858,0.830608,-0.669756,0.898620,-0.098512,-0.128774,-0.055798,-0.078325,0.356981,0.338043,...,-0.329125,-0.479140,-0.211689,-0.058756,2.244145,1.337896,-0.16555,-0.165479,-0.288988,0.0
257668,-0.197857,0.830608,-0.669756,0.898620,-0.098512,-0.128774,-0.055798,-0.078325,0.027070,0.019533,...,-0.329125,-0.479140,-0.211689,-0.058756,1.396394,0.797006,-0.16555,-0.165479,-0.288988,0.0
257670,-0.197857,0.830608,-0.669756,0.898620,-0.098512,-0.128774,-0.055798,-0.078325,0.027070,0.019533,...,-0.329125,-0.479140,-0.211689,-0.058756,-0.581691,-0.555217,-0.16555,-0.165479,-0.288988,0.0
257671,-0.197857,0.830608,-0.669756,0.898620,-0.098512,-0.128774,-0.055798,-0.078325,0.027070,0.019533,...,-0.329125,-0.479140,-0.211689,-0.058756,1.961562,0.932229,-0.16555,-0.165479,-0.288988,0.0


In [62]:
# Split Data into Features (X) and Target (y)
# label_encoder = LabelEncoder()

# df['label'] = label_encoder.fit_transform(df['label'])
# X = df.drop(['label'], axis=1)
# y = df['label']

# Saving column names
original_feature_names = X.columns

# Split Data into Training and Testing Sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Split X_train and y_train to create a validation set
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.25, random_state=42)  # 0.25 x 0.8 = 0.2

# Initialize the StandardScaler
scaler = StandardScaler()

# Fit on training set only
X_train = scaler.fit_transform(X_train)

# Apply transform to the test set
X_test = scaler.transform(X_test)

# Apply transform to the validation and test sets
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

X_train.shape
joblib.dump(scaler,'labelling.joblib')

C:\Python312\Lib\site-packages\sklearn\base.py:493: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


['labelling.joblib']

In [59]:
model = LogisticRegression(max_iter=1000).fit(X_train, y_train)

# Predict on validation set
y_val_predictions = model.predict(X_val)

# Predict on test set
y_test_predictions = model.predict(X_test)
 
# Calculate performance metrics for validation set
accuracy_val = accuracy_score(y_val, y_val_predictions)
recall_val = recall_score(y_val, y_val_predictions, average='weighted')
precision_val = precision_score(y_val, y_val_predictions, average='weighted')
f1s_val = f1_score(y_val, y_val_predictions, average='weighted')

# Calculate performance metrics for test set
accuracy_test = accuracy_score(y_test, y_test_predictions)
recall_test = recall_score(y_test, y_test_predictions, average='weighted')
precision_test = precision_score(y_test, y_test_predictions, average='weighted')
f1s_test = f1_score(y_test, y_test_predictions, average='weighted')

# Print performance metrics for validation set
print("Validation Set Performance:")
print("Accuracy: " + "{:.2%}".format(accuracy_val))
print("Recall: " + "{:.2%}".format(recall_val))
print("Precision: " + "{:.2%}".format(precision_val))
print("F1-Score: " + "{:.2%}".format(f1s_val))

# Print performance metrics for test set
print("\nTest Set Performance:")
print("Accuracy: " + "{:.2%}".format(accuracy_test))
print("Recall: " + "{:.2%}".format(recall_test))
print("Precision: " + "{:.2%}".format(precision_test))
print("F1-Score: " + "{:.2%}".format(f1s_test))


Validation Set Performance:
Accuracy: 94.63%
Recall: 94.63%
Precision: 94.58%
F1-Score: 94.57%

Test Set Performance:
Accuracy: 94.41%
Recall: 94.41%
Precision: 94.36%
F1-Score: 94.32%


In [22]:
joblib.dump(model,'logistic.joblib')

['logistic.joblib']